# Credit Default Modelling

This notebook builds and evaluates models for predicting customer default.

In [36]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import log_loss
from sklearn.base import clone

from scipy.optimize import minimize_scalar

from xgboost import XGBClassifier
from catboost import CatBoostClassifier

train = pd.read_csv("../data/raw/train.csv")
test = pd.read_csv("../data/raw/test.csv")

Separate the predictor variables from the default target and remove client_id.

In [3]:
X = train.drop(columns=["default", "client_id"])
y = train["default"]

X_test = test.drop(columns=["client_id"])

Split the labelled data into training and validation sets. Use stratification to keep the default rate similar in both sets.


In [4]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=50
)


Create lists of columns that are categorical and standardise the remaining numerical features.


In [5]:
categorical_cols = [
    "SEX",
    "EDUCATION",
    "MARRIAGE",
    "PAY_0",
    "PAY_2",
    "PAY_3",
    "PAY_4",
    "PAY_5",
    "PAY_6"
]

numeric_cols = [
    col for col in X.columns
    if col not in categorical_cols
]


In [6]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_cols
        ),
        (
            "num",
            StandardScaler(),
            numeric_cols
        )
    ]
)


Use logistic regression as the first baseline model.


In [7]:
model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])


In [8]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=50
)

cv_scores = -cross_val_score(
    model,
    X,
    y,
    cv=cv,
    scoring="neg_log_loss"
)

baseline_logloss = cv_scores.mean()

print("Fold Log Loss:", cv_scores)
print("Mean CV Log Loss:", baseline_logloss)
print("Standard Deviation:", cv_scores.std())

Fold Log Loss: [0.43757934 0.42918817 0.44323252 0.43680413 0.43591197]
Mean CV Log Loss: 0.43654322437846715
Standard Deviation: 0.004480704353575942


Use XGBoost as a stronger nonlinear model and evaluate it using the same 5-fold cross-validation setup as the logistic regression baseline.

In [9]:
boost_categorical_cols = [
    "SEX",
    "EDUCATION",
    "MARRIAGE"
]

In [10]:
xgb_preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            boost_categorical_cols
        )
    ],
    remainder="passthrough"
)

In [11]:
xgb_model = Pipeline([
    ("preprocessor", xgb_preprocessor),
    (
        "classifier",
        XGBClassifier(
            n_estimators=500,
            learning_rate=0.03,
            max_depth=4,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="binary:logistic",
            eval_metric="logloss",
            random_state=50,
            n_jobs=-1
        )
    )
])

In [12]:
xgb_scores = -cross_val_score(
    xgb_model,
    X,
    y,
    cv=cv,
    scoring="neg_log_loss"
)

xgb_logloss = xgb_scores.mean()

print("Fold Log Loss:", xgb_scores)
print("Mean CV Log Loss:", xgb_logloss)
print("Standard Deviation:", xgb_scores.std())

Fold Log Loss: [0.42739013 0.41644689 0.43619543 0.43129441 0.42460811]
Mean CV Log Loss: 0.4271869957447052
Standard Deviation: 0.006634221762947777


Use CatBoost as another stronger nonlinear model and evaluate it using the same 5-fold cross-validation setup as the previous models.

In [13]:
cat_model = CatBoostClassifier(
    iterations=500,
    learning_rate=0.03,
    depth=5,
    loss_function="Logloss",
    cat_features=boost_categorical_cols,
    verbose=0,
    random_seed=50,
    thread_count=-1
)

In [14]:
cat_scores = []

for fold, (train_idx, valid_idx) in enumerate(cv.split(X, y), start=1):

    X_train_fold = X.iloc[train_idx]
    X_valid_fold = X.iloc[valid_idx]

    y_train_fold = y.iloc[train_idx]
    y_valid_fold = y.iloc[valid_idx]

    cat_model = CatBoostClassifier(
        iterations=500,
        learning_rate=0.03,
        depth=5,
        loss_function="Logloss",
        verbose=0,
        random_seed=50,
        thread_count=-1
    )

    cat_model.fit(
        X_train_fold,
        y_train_fold,
        cat_features=boost_categorical_cols
    )

    cat_probs = cat_model.predict_proba(X_valid_fold)[:, 1]

    fold_score = log_loss(
        y_valid_fold,
        cat_probs
    )

    cat_scores.append(fold_score)

    print(f"Fold {fold}: {fold_score:.5f}")

Fold 1: 0.42671
Fold 2: 0.41662
Fold 3: 0.43706
Fold 4: 0.42861
Fold 5: 0.42420


In [15]:
cat_scores = np.array(cat_scores)

cat_logloss = cat_scores.mean()

print("\nFold Log Loss:", cat_scores)
print("Mean CV Log Loss:", cat_logloss)
print("Standard Deviation:", cat_scores.std())


Fold Log Loss: [0.4267056  0.4166158  0.43706394 0.42861445 0.42420285]
Mean CV Log Loss: 0.42664053018775033
Standard Deviation: 0.006617939412801522


Compare results of models so far

In [16]:
results = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "XGBoost",
        "CatBoost"
    ],
    "Mean CV Log Loss": [
        baseline_logloss,
        xgb_logloss,
        cat_logloss
    ]
})

results

,Model,Mean CV Log Loss
0,Logistic Regression,0.436543
1,XGBoost,0.427187
2,CatBoost,0.426641


Retrain all models again but with feature engineering

In [19]:
pay_cols = [
    "PAY_0", "PAY_2", "PAY_3",
    "PAY_4", "PAY_5", "PAY_6"
]

bill_cols = [
    "BILL_AMT1", "BILL_AMT2", "BILL_AMT3",
    "BILL_AMT4", "BILL_AMT5", "BILL_AMT6"
]

payment_cols = [
    "PAY_AMT1", "PAY_AMT2", "PAY_AMT3",
    "PAY_AMT4", "PAY_AMT5", "PAY_AMT6"
]

In [20]:
def add_features(df):
    df = df.copy()

    df["delayed_months"] = (df[pay_cols] > 0).sum(axis=1)
    df["max_delay"] = df[pay_cols].max(axis=1)
    df["avg_repayment_status"] = df[pay_cols].mean(axis=1)

    df["avg_bill"] = df[bill_cols].mean(axis=1)
    df["avg_payment"] = df[payment_cols].mean(axis=1)

    df["recent_utilisation"] = (
        df["BILL_AMT1"] / df["LIMIT_BAL"]
    )

    return df

In [21]:
X_fe = add_features(X)
X_test_fe = add_features(X_test)

In [22]:
xgb_fe_scores = -cross_val_score(
    xgb_model,
    X_fe,
    y,
    cv=cv,
    scoring="neg_log_loss"
)

xgb_fe_logloss = xgb_fe_scores.mean()

print("Fold Log Loss:", xgb_fe_scores)
print("Mean CV Log Loss:", xgb_fe_logloss)
print("Standard Deviation:", xgb_fe_scores.std())

Fold Log Loss: [0.42556357 0.41657162 0.4350349  0.4312329  0.42347962]
Mean CV Log Loss: 0.4263765215873718
Standard Deviation: 0.006383432178775376


In [23]:
xgb_fe_scores = -cross_val_score(
    xgb_model,
    X_fe,
    y,
    cv=cv,
    scoring="neg_log_loss"
)

xgb_fe_logloss = xgb_fe_scores.mean()

print("Fold Log Loss:", xgb_fe_scores)
print("Mean CV Log Loss:", xgb_fe_logloss)
print("Standard Deviation:", xgb_fe_scores.std())

Fold Log Loss: [0.42556357 0.41657162 0.4350349  0.4312329  0.42347962]
Mean CV Log Loss: 0.4263765215873718
Standard Deviation: 0.006383432178775376


In [24]:
print("Raw XGBoost:", xgb_logloss)
print("XGBoost + Features:", xgb_fe_logloss)

Raw XGBoost: 0.4271869957447052
XGBoost + Features: 0.4263765215873718


In [25]:
cat_fe_scores = []

for fold, (train_idx, valid_idx) in enumerate(cv.split(X_fe, y), start=1):

    X_train_fold = X_fe.iloc[train_idx]
    X_valid_fold = X_fe.iloc[valid_idx]

    y_train_fold = y.iloc[train_idx]
    y_valid_fold = y.iloc[valid_idx]

    cat_model = CatBoostClassifier(
        iterations=500,
        learning_rate=0.03,
        depth=5,
        loss_function="Logloss",
        verbose=0,
        random_seed=50,
        thread_count=-1
    )

    cat_model.fit(
        X_train_fold,
        y_train_fold,
        cat_features=boost_categorical_cols
    )

    cat_probs = cat_model.predict_proba(X_valid_fold)[:, 1]

    fold_score = log_loss(y_valid_fold, cat_probs)

    cat_fe_scores.append(fold_score)

    print(f"Fold {fold}: {fold_score:.5f}")

Fold 1: 0.42388
Fold 2: 0.41537
Fold 3: 0.43531
Fold 4: 0.42825
Fold 5: 0.42204


In [26]:
cat_fe_scores = np.array(cat_fe_scores)

cat_fe_logloss = cat_fe_scores.mean()

print("\nFold Log Loss:", cat_fe_scores)
print("Mean CV Log Loss:", cat_fe_logloss)
print("Standard Deviation:", cat_fe_scores.std())


Fold Log Loss: [0.42387969 0.41536903 0.43530678 0.42824819 0.42203554]
Mean CV Log Loss: 0.4249678481708238
Standard Deviation: 0.006626822834946325


Ensemble

In [28]:
xgb_oof = np.zeros(len(X_fe))
cat_oof = np.zeros(len(X_fe))

In [31]:
for fold, (train_idx, valid_idx) in enumerate(cv.split(X_fe, y), start=1):

    X_train_fold = X_fe.iloc[train_idx]
    X_valid_fold = X_fe.iloc[valid_idx]

    y_train_fold = y.iloc[train_idx]

    # XGBoost
    xgb_fold = clone(xgb_model)

    xgb_fold.fit(
        X_train_fold,
        y_train_fold
    )

    xgb_oof[valid_idx] = (
        xgb_fold.predict_proba(X_valid_fold)[:, 1]
    )

    # CatBoost
    cat_fold = CatBoostClassifier(
        iterations=500,
        learning_rate=0.03,
        depth=5,
        loss_function="Logloss",
        verbose=0,
        random_seed=50,
        thread_count=-1
    )

    cat_fold.fit(
        X_train_fold,
        y_train_fold,
        cat_features=boost_categorical_cols
    )

    cat_oof[valid_idx] = (
        cat_fold.predict_proba(X_valid_fold)[:, 1]
    )

    print(f"Finished Fold {fold}")

Finished Fold 1
Finished Fold 2
Finished Fold 3
Finished Fold 4
Finished Fold 5


In [32]:
print("XGBoost OOF:", log_loss(y, xgb_oof))
print("CatBoost OOF:", log_loss(y, cat_oof))

XGBoost OOF: 0.42637652059796993
CatBoost OOF: 0.4249678481708238


In [33]:
blend_oof = (
    0.5 * xgb_oof +
    0.5 * cat_oof
)

blend_logloss = log_loss(
    y,
    blend_oof
)

print("50/50 Blend Log Loss:", blend_logloss)

50/50 Blend Log Loss: 0.4249496225716023


In [37]:
def blend_loss(w):
    blended = (
        w * xgb_oof +
        (1 - w) * cat_oof
    )
    
    return log_loss(y, blended)

result = minimize_scalar(
    blend_loss,
    bounds=(0, 1),
    method="bounded"
)

best_weight = result.x
best_blend_logloss = result.fun

print("Best XGBoost Weight:", best_weight)
print("Best CatBoost Weight:", 1 - best_weight)
print("Best Blend Log Loss:", best_blend_logloss)

Best XGBoost Weight: 0.25644704529717677
Best CatBoost Weight: 0.7435529547028232
Best Blend Log Loss: 0.4247826745877479


Tune models

In [38]:
cat_param_sets = [
    {"iterations": 500, "learning_rate": 0.03, "depth": 4},
    {"iterations": 500, "learning_rate": 0.03, "depth": 5},
    {"iterations": 500, "learning_rate": 0.03, "depth": 6},
    {"iterations": 700, "learning_rate": 0.02, "depth": 5},
    {"iterations": 700, "learning_rate": 0.02, "depth": 6},
    {"iterations": 400, "learning_rate": 0.05, "depth": 5},
]

In [39]:
cat_tuning_results = []

for params in cat_param_sets:

    fold_scores = []

    for train_idx, valid_idx in cv.split(X_fe, y):

        X_train_fold = X_fe.iloc[train_idx]
        X_valid_fold = X_fe.iloc[valid_idx]

        y_train_fold = y.iloc[train_idx]
        y_valid_fold = y.iloc[valid_idx]

        model = CatBoostClassifier(
            iterations=params["iterations"],
            learning_rate=params["learning_rate"],
            depth=params["depth"],
            loss_function="Logloss",
            verbose=0,
            random_seed=50,
            thread_count=-1
        )

        model.fit(
            X_train_fold,
            y_train_fold,
            cat_features=boost_categorical_cols
        )

        probs = model.predict_proba(X_valid_fold)[:, 1]

        fold_scores.append(
            log_loss(y_valid_fold, probs)
        )

    cat_tuning_results.append({
        **params,
        "Mean CV Log Loss": np.mean(fold_scores),
        "Standard Deviation": np.std(fold_scores)
    })

In [40]:
cat_tuning_results = pd.DataFrame(
    cat_tuning_results
).sort_values("Mean CV Log Loss")

cat_tuning_results

,iterations,learning_rate,depth,Mean CV Log Loss,Standard Deviation
4,700,0.02,6,0.424350,0.006195
2,500,0.03,6,0.424498,0.006405
3,700,0.02,5,0.424524,0.006292
0,500,0.03,4,0.424872,0.006228
1,500,0.03,5,0.424968,0.006627
5,400,0.05,5,0.425337,0.006270


In [41]:
xgb_param_sets = [
    {"n_estimators": 500, "learning_rate": 0.03, "max_depth": 3},
    {"n_estimators": 500, "learning_rate": 0.03, "max_depth": 4},
    {"n_estimators": 500, "learning_rate": 0.03, "max_depth": 5},
    {"n_estimators": 700, "learning_rate": 0.02, "max_depth": 4},
    {"n_estimators": 700, "learning_rate": 0.02, "max_depth": 5},
    {"n_estimators": 400, "learning_rate": 0.05, "max_depth": 4},
]

In [42]:
xgb_tuning_results = []

for params in xgb_param_sets:

    tuned_xgb = Pipeline([
        ("preprocessor", xgb_preprocessor),

        ("classifier", XGBClassifier(
            n_estimators=params["n_estimators"],
            learning_rate=params["learning_rate"],
            max_depth=params["max_depth"],
            subsample=0.8,
            colsample_bytree=0.8,
            objective="binary:logistic",
            eval_metric="logloss",
            random_state=50,
            n_jobs=-1
        ))
    ])

    scores = -cross_val_score(
        tuned_xgb,
        X_fe,
        y,
        cv=cv,
        scoring="neg_log_loss"
    )

    xgb_tuning_results.append({
        **params,
        "Mean CV Log Loss": scores.mean(),
        "Standard Deviation": scores.std()
    })

In [43]:
xgb_tuning_results = (
    pd.DataFrame(xgb_tuning_results)
    .sort_values("Mean CV Log Loss")
)

xgb_tuning_results

,n_estimators,learning_rate,max_depth,Mean CV Log Loss,Standard Deviation
0,500,0.03,3,0.425379,0.006144
3,700,0.02,4,0.425813,0.006173
1,500,0.03,4,0.426377,0.006383
4,700,0.02,5,0.426937,0.006029
2,500,0.03,5,0.427342,0.006357
5,400,0.05,4,0.427693,0.006354


In [44]:
tuned_xgb_oof = np.zeros(len(X_fe))
tuned_cat_oof = np.zeros(len(X_fe))

for fold, (train_idx, valid_idx) in enumerate(cv.split(X_fe, y), start=1):

    X_train_fold = X_fe.iloc[train_idx]
    X_valid_fold = X_fe.iloc[valid_idx]

    y_train_fold = y.iloc[train_idx]

    # Tuned XGBoost
    tuned_xgb = Pipeline([
        ("preprocessor", xgb_preprocessor),

        ("classifier", XGBClassifier(
            n_estimators=500,
            learning_rate=0.03,
            max_depth=3,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="binary:logistic",
            eval_metric="logloss",
            random_state=50,
            n_jobs=-1
        ))
    ])

    tuned_xgb.fit(
        X_train_fold,
        y_train_fold
    )

    tuned_xgb_oof[valid_idx] = (
        tuned_xgb.predict_proba(X_valid_fold)[:, 1]
    )

    # Tuned CatBoost
    tuned_cat = CatBoostClassifier(
        iterations=700,
        learning_rate=0.02,
        depth=6,
        loss_function="Logloss",
        verbose=0,
        random_seed=50,
        thread_count=-1
    )

    tuned_cat.fit(
        X_train_fold,
        y_train_fold,
        cat_features=boost_categorical_cols
    )

    tuned_cat_oof[valid_idx] = (
        tuned_cat.predict_proba(X_valid_fold)[:, 1]
    )

    print(f"Finished Fold {fold}")

Finished Fold 1
Finished Fold 2
Finished Fold 3
Finished Fold 4
Finished Fold 5


In [45]:
print(
    "Tuned XGBoost OOF:",
    log_loss(y, tuned_xgb_oof)
)

print(
    "Tuned CatBoost OOF:",
    log_loss(y, tuned_cat_oof)
)

Tuned XGBoost OOF: 0.42537881200923705
Tuned CatBoost OOF: 0.4243504197171851


In [46]:
def tuned_blend_loss(w):

    blended = (
        w * tuned_xgb_oof +
        (1 - w) * tuned_cat_oof
    )

    return log_loss(y, blended)


result = minimize_scalar(
    tuned_blend_loss,
    bounds=(0, 1),
    method="bounded"
)

tuned_xgb_weight = result.x
tuned_cat_weight = 1 - result.x
tuned_blend_logloss = result.fun

print("XGBoost Weight:", tuned_xgb_weight)
print("CatBoost Weight:", tuned_cat_weight)
print("Tuned Blend Log Loss:", tuned_blend_logloss)

XGBoost Weight: 0.2128142851065431
CatBoost Weight: 0.7871857148934569
Tuned Blend Log Loss: 0.42426614911134675


In [48]:
# Tuned XGBoost
final_xgb = Pipeline([
    ("preprocessor", xgb_preprocessor),

    ("classifier", XGBClassifier(
        n_estimators=500,
        learning_rate=0.03,
        max_depth=3,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=50,
        n_jobs=-1
    ))
])

final_xgb.fit(X_fe, y)

xgb_test_pred = final_xgb.predict_proba(X_test_fe)[:, 1]

In [49]:
final_cat = CatBoostClassifier(
    iterations=700,
    learning_rate=0.02,
    depth=6,
    loss_function="Logloss",
    verbose=0,
    random_seed=50,
    thread_count=-1
)

final_cat.fit(
    X_fe,
    y,
    cat_features=boost_categorical_cols
)

cat_test_pred = final_cat.predict_proba(X_test_fe)[:, 1]

In [50]:
final_pred = (
    0.2128142851 * xgb_test_pred +
    0.7871857149 * cat_test_pred
)

In [53]:
submission = pd.DataFrame({
    "client_id": test["client_id"],
    "default": final_pred
})

submission.to_csv(
    "../submission/submission_tuned_ensemble.csv",
    index=False
)

submission.head()

,client_id,default
0,CC_0012A082B7B7,0.182866
1,CC_0012BC27DFB6,0.157255
2,CC_001563B2143D,0.113074
3,CC_001B46930B6F,0.093141
4,CC_001D771E1C9F,0.053340
